# Lab 2 – Reuse and Orchestrate AI Agents with Microsoft Foundry SDK (Python)

## Scenario
You will reuse the **ProductInventoryManager** agent created in Lab 1 and interact with it from **Visual Studio Code** using **Python**. Then you will create a second agent and orchestrate both agents using the **Microsoft Agent Framework** workflow pattern.

> **Important:** In the new Foundry Agents architecture, agent behavior is defined at creation time (versions are static). Avoid trying to override tools/instructions at runtime.

## Prerequisites
- Completed **Lab 1** (you have **Project Endpoint** + **ProductInventoryManager Agent ID**)
- Visual Studio Code + Microsoft Foundry extension installed
- Python 3.10+
- Azure CLI installed and authenticated: `az login`

### Values you need from Lab 1
- `AZURE_FOUNDRY_PROJECT_ENDPOINT`
- `PRODUCT_INVENTORY_MANAGER_AGENT_ID`

## Step 1 — (Optional) Connect VS Code to Foundry
1. Install the **Microsoft Foundry for VS Code** extension.
2. Sign in to Azure.
3. Set your **default Foundry project** so Agents/Models appear in the sidebar.

This step validates you are pointing at the same Foundry project as Lab 1.

## Step 2 — Create a Python environment
Run these commands in the VS Code terminal.

In [ ]:
python -m venv .venv
# Windows:
# .venv\Scripts\activate
# macOS/Linux:
# source .venv/bin/activate

python -m pip install --upgrade pip

## Step 3 — Install required packages
These packages are used in Microsoft Foundry quickstarts and SDK guidance for working with Foundry projects and agents.

In [ ]:
pip install azure-ai-projects --pre
pip install azure-identity openai python-dotenv

# Microsoft Agent Framework (Python)
pip install agent-framework --pre

## Step 4 — Configure environment variables
Create a `.env` file in this folder with the following values.

In [ ]:
# Create/overwrite .env with your values
# (You can also set these in your shell instead.)

AZURE_FOUNDRY_PROJECT_ENDPOINT="<paste-your-project-endpoint-here>"
PRODUCT_INVENTORY_MANAGER_AGENT_ID="<paste-agent-id-here>"

# Optional if you create a new agent in this lab:
AZURE_FOUNDRY_PROJECT_MODEL_ID="gpt-4o-mini"

## Step 5 — Authenticate to Azure
Make sure you are logged in before running Python.

In [ ]:
# Run in terminal (not in notebook)
# az login

## Step 6 — Load config + connect to your Foundry project (Python)
This cell loads `.env` and creates a Foundry project client.

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()

project_endpoint = os.environ["AZURE_FOUNDRY_PROJECT_ENDPOINT"]
existing_agent_id = os.environ["PRODUCT_INVENTORY_MANAGER_AGENT_ID"]
model_id = os.getenv("AZURE_FOUNDRY_PROJECT_MODEL_ID", "gpt-4o-mini")

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)

print("Connected to project endpoint:", project_endpoint)
print("Reusing agent id:", existing_agent_id)
print("Default model id (for new agents):", model_id)

## Step 7 — Reuse the ProductInventoryManager agent by ID
This cell retrieves the persistent agent created in Lab 1.

In [ ]:
# The projects client exposes agent operations via .agents
# Retrieve existing agent
product_inventory_agent = project_client.agents.get(agent_id=existing_agent_id)

print("Agent name:", getattr(product_inventory_agent, 'name', None))
print("Agent id:", getattr(product_inventory_agent, 'id', None))

## Step 8 — Ask the existing agent a question
This uses the OpenAI-compatible Responses API through the Foundry project.

> Note: Depending on SDK version, you may use `project_client.inference.get_openai_client()` or a similar helper. If your installed SDK exposes a different helper, use the code sample generated by Foundry/VS Code for your agent.

In [ ]:
# --- Minimal pattern using OpenAI-compatible client ---
# The AI Projects SDK provides a helper to get an authenticated OpenAI client for the project.
# If your version differs, use the code snippet from: Foundry portal/VS Code -> Agent -> Open code file.

openai_client = project_client.inference.get_openai_client()

question = "Describe our top-selling product and give a short product description."

response = openai_client.responses.create(
    input=question,
    extra_body={"agent": {"type": "agent_reference", "id": existing_agent_id}},
)

print(response.output_text)

## Step 9 — Create a second agent: SalesInsightsAgent
This cell creates a new persistent agent in your Foundry project.

In [ ]:
sales_instructions = (
    "You are SalesInsightsAgent. Analyze sales-related questions and produce concise, executive-friendly insights. "
    "Focus on trends, top movers, and actionable next steps. If data is missing, state assumptions."
)

# Create the agent (persistent)
sales_agent = project_client.agents.create(
    name="SalesInsightsAgent",
    model=model_id,
    instructions=sales_instructions,
)

print("Created SalesInsightsAgent id:", getattr(sales_agent, 'id', None))

## Step 10 — Test SalesInsightsAgent
Ask a sales-focused question.

In [ ]:
sales_agent_id = getattr(sales_agent, 'id', None)

question = "Which products appear most popular and what are 3 talking points for leadership?"

response = openai_client.responses.create(
    input=question,
    extra_body={"agent": {"type": "agent_reference", "id": sales_agent_id}},
)

print(response.output_text)

## Step 11 — Orchestrate both agents with Microsoft Agent Framework (Python)
We will build a simple **sequential** workflow:
1) ProductInventoryManager answers inventory/product context
2) SalesInsightsAgent refines into sales insights

In Agent Framework, workflows are graph-based and you can wrap them as an agent for a unified interface.

In [ ]:
# Agent Framework imports
from agent_framework import WorkflowBuilder
from agent_framework.azure_ai import FoundryPersistentAgent

# Wrap Foundry agents as Agent Framework agents
inv_agent = FoundryPersistentAgent(project_endpoint=project_endpoint, agent_id=existing_agent_id, credential=credential)
sales_agent_af = FoundryPersistentAgent(project_endpoint=project_endpoint, agent_id=sales_agent_id, credential=credential)

# Build sequential workflow: inv_agent -> sales_agent_af
workflow = (
    WorkflowBuilder(start_executor=inv_agent)
    .add_edge(inv_agent, sales_agent_af)
    .build()
)

prompt = "We need a one-paragraph overview of the top product plus 3 sales insights."
result = await workflow.run_async(prompt)
print(result.get_outputs())

## Step 12 — Cleanup (optional but recommended)
If you created SalesInsightsAgent only for the lab, delete it to avoid resource sprawl/cost.

In [ ]:
# Delete the agent you created in this lab (optional)
# project_client.agents.delete(agent_id=sales_agent_id)
# print('Deleted SalesInsightsAgent:', sales_agent_id)

## Validation Checklist
- ✅ Connected to Foundry project from Python
- ✅ Reused ProductInventoryManager by ID
- ✅ Created and tested SalesInsightsAgent
- ✅ Ran a sequential workflow orchestrating both agents